### A5.1.3. Build Thread Safe Queue Class from Scratch

**Problem:**

Build a thread-safe queue that supports `put` and `get` operations from multiple threads simultaneously, with blocking behavior when the queue is empty.

**Explanation:**

A thread-safe queue wraps a regular collection (like a `deque`) with synchronization primitives so multiple threads can add and remove items without corruption.

How to build it:

1. Store a `collections.deque` as the internal buffer in `__init__`.
2. Create a `threading.Lock` to protect access to the buffer.
3. Create a `threading.Condition` (wrapping the lock) so that `get` can wait when the queue is empty.
4. `put(item)`: acquire the lock, append the item to the deque, then notify one waiting thread via the condition.
5. `get()`: acquire the lock, wait on the condition while the deque is empty, then popleft and return the item.

**Example:**

One thread calls `queue.put(42)`. Another thread blocked on `queue.get()` wakes up and receives `42`.

In [ ]:
import threading
from collections import deque


class ThreadSafeQueue:
    def __init__(self):
        self.buffer = deque()
        self.lock = threading.Lock()
        self.not_empty = threading.Condition(self.lock)

    def put(self, item):
        with self.not_empty:
            self.buffer.append(item)
            self.not_empty.notify()

    def get(self):
        with self.not_empty:
            while not self.buffer:
                self.not_empty.wait()
            return self.buffer.popleft()


queue = ThreadSafeQueue()
results = []


def producer():
    for index in range(5):
        queue.put(index)


def consumer():
    for _ in range(5):
        results.append(queue.get())


producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

producer_thread.start()
consumer_thread.start()

producer_thread.join()
consumer_thread.join()

print(f"Consumed: {results}")

**References:**

📘 Python Documentation — [threading.Condition](https://docs.python.org/3/library/threading.html#condition-objects)

📘 Python Documentation — [collections.deque](https://docs.python.org/3/library/collections.html#collections.deque)

---

[⬅️ Previous: Build Streaming Tokenizer Class from Scratch](./02_build_streaming_tokenizer_class_from_scratch.ipynb) | [Next: Build Object Pool Class from Scratch ➡️](./04_build_object_pool_class_from_scratch.ipynb)